# 01 Parking rules
What the register contains, what parsed cleanly, and what did not.

Run `python pipeline/fetch.py` then `python pipeline/process.py` first.

In [ ]:
import geopandas as gpd, pandas as pd, matplotlib.pyplot as plt

areas = gpd.read_parquet("../data/processed/parking_rules.parquet")  # 8,754 areas, metres
print(len(areas), "areas | CRS", areas.crs.to_epsg())

## How much can the app actually answer?

In [ ]:
pd.crosstab(areas["rule_type"], areas["status"], margins=True)

## What stopped an area being official?
The parsers fail closed: anything they cannot fully read becomes a code plus a reason in plain words.

In [ ]:
areas["issue_codes"].dropna().str.split(",").explode().value_counts()

In [ ]:
areas.loc[areas["reason"].notna(), ["voimassaolo", "kesto", "tyyppi", "reason"]].head(10)

## Raw values behind the parsers
A new spelling from the city shows up here first.

In [ ]:
for col in ["voimassaolo", "kesto", "kausi"]:
    print(col, "-", areas[col].nunique(), "distinct")
    print(areas[col].value_counts().head(8).to_string(), "\n")

## Where the gaps are
Missing hours cluster outside the city centre.

In [ ]:
ax = areas.plot(column="status", legend=True, figsize=(9, 9), linewidth=2)
ax.set_title("Rule status per parking area"); ax.set_axis_off()

## Roadworks
Areas under a temporary traffic arrangement cannot be trusted while it lasts.

In [ ]:
areas["roadworks_until"].notna().sum()